# Figures KPI
## Phân tích phân bố quãng đường và lượng chuyến đi theo giờ


In [ ]:
import warnings
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.simplefilter('ignore')
sns.set(style='whitegrid', palette='deep')

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'Data').exists():
    ROOT = ROOT.parent

CLEAN_DIR = ROOT / 'Data' / 'processed' / 'clean_data'
FIGURE_DIR = ROOT / 'Data' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_clean_data():
    clean_files = sorted(CLEAN_DIR.glob('clean_2021-*.parquet'))
    columns = ['tpep_pickup_datetime', 'trip_distance', 'total_amount']
    df_list = [pd.read_parquet(path, columns=columns) for path in clean_files]
    df = pd.concat(df_list, ignore_index=True)
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
    df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
    return df

clean_df = load_clean_data()
clean_df.head()

## Bảng phân bố quãng đường di chuyển


In [ ]:
plt.figure(figsize=(12, 6))
ax = sns.histplot(clean_df['trip_distance'].clip(upper=50), bins=80, kde=False, color='#2a9d8f')
ax.set_title('Phân bố quãng đường di chuyển (km) - Yellow Taxi 2021', fontsize=16)
ax.set_xlabel('Trip Distance (km)')
ax.set_ylabel('Số chuyến đi')
ax.set_ylim(0, 1_200_000)
ax.set_xlim(0, 50)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'trip_distance_distribution.png', dpi=200)
plt.show()

## Biểu đồ đường lượng chuyến đi theo giờ
Tính tổng số chuyến đi theo giờ và hiển thị dưới dạng biểu đồ đường.

In [ ]:
hourly_counts = clean_df.groupby('pickup_hour').size().reindex(range(24), fill_value=0).rename('trip_count')
plt.figure(figsize=(14, 6))
ax = hourly_counts.plot(linewidth=1, color='#e76f51')
ax.set_title('Lượng chuyến đi theo giờ - Yellow Taxi 2021', fontsize=16)
ax.set_xlabel('Giờ khởi hành')
ax.set_ylabel('Số chuyến đi')
ax.set_ylim(0, int(hourly_counts.max() * 1.05))
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'trips_by_hour_line.png', dpi=200)
plt.show()

## Tóm tắt KPI
Hiển thị một số chỉ số chính lấy từ dữ liệu sạch.

In [ ]:
summary = {
    'Tổng số chuyến đi': len(clean_df),
    'Quãng đường trung bình (km)': clean_df['trip_distance'].mean(),
    'Quãng đường trung vị (km)': clean_df['trip_distance'].median(),
    'Tổng doanh thu (USD)': clean_df['total_amount'].sum(),
    'Giờ có nhiều chuyến nhất': hourly_counts.idxmax(),
    'Số chuyến giờ cao điểm': int(hourly_counts.max())
}
summary